# Section 2: K-Nearest Neighbor (K-NN) Classifier

In this section, we build our first concrete implementation of the data-driven approach: the **K-Nearest Neighbor (K-NN)** algorithm. Though rarely used in modern computer vision due to significant drawbacks, K-NN introduces critical machine learning concepts such as **distance metrics**, **hyperparameters**, and **decision boundaries**.

## 2.1 Concept and Algorithm

The K-NN algorithm is beautifully simple. It doesn't attempt to learn a complex mathematical function mapping an image to a label. Instead, it relies on brute-force memorization and comparison.

*   **Train Function:** Simply memorize every single image and its corresponding label in the training dataset.
*   **Predict Function:** Given a new, unseen test image, compare it against *every* training image. Find the top $K$ most similar training images (the "nearest neighbors"), and have them vote on the label. The majority vote becomes the prediction.

### Computational Complexity
Let's analyze the time complexity, assuming we have $N$ training examples.

*   **Training Time:** $O(1)$ — We are simply copying data into memory. No actual computation happens.
*   **Prediction Time:** $O(N)$ — For *each* new test image, we must calculate the distance to all $N$ training images.

> [!WARNING]
> **Major Pitfall:** This complexity is completely backwards for real-world applications! In practice, we want models that are extremely fast at prediction time (e.g., real-time inference on a self-driving car). We don't care if training takes a week in an offline data center, but prediction must be instant. K-NN fails this requirement entirely.

In [ ]:
import numpy as np
import torch

class NearestNeighborNumpy:
    def __init__(self):
        pass
        
    def train(self, X: np.ndarray, y: np.ndarray):
        """
        X is N x D where each row is an example. Y is 1-dimension of size N
        O(1) Training time!
        """
        self.X_train = X
        self.y_train = y
        
    def predict(self, X: np.ndarray, distance_metric='L1'):
        """
        O(N) Prediction time!
        """
        num_test = X.shape[0]
        Ypred = np.zeros(num_test, dtype=self.y_train.dtype)
        
        # In a real implementation, we would vectorize this. 
        # But for intuition, here is the explicit loop:
        for i in range(num_test):
            if distance_metric == 'L1':
                distances = np.sum(np.abs(self.X_train - X[i,:]), axis=1)
            else: # L2
                distances = np.sqrt(np.sum(np.square(self.X_train - X[i,:]), axis=1))
                
            min_index = np.argmin(distances)
            Ypred[i] = self.y_train[min_index]
            
        return Ypred

# PyTorch Equivalent (utilizing GPU for faster distance computation if available)
class NearestNeighborPyTorch:
    def __init__(self):
        self.X_train = None
        self.y_train = None
        
    def train(self, X: torch.Tensor, y: torch.Tensor):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.X_train = X.to(self.device)
        self.y_train = y.to(self.device)
        
    def predict(self, X: torch.Tensor):
        X = X.to(self.device)
        num_test = X.shape[0]
        Ypred = torch.zeros(num_test, dtype=self.y_train.dtype, device=self.device)
        
        # Efficient vectorized L1 distance using PyTorch broadcasting
        for i in range(num_test):
            distances = torch.sum(torch.abs(self.X_train - X[i,:]), dim=1)
            min_index = torch.argmin(distances)
            Ypred[i] = self.y_train[min_index]
            
        return Ypred.cpu()

## 2.2 Distance Metrics

To find the "nearest" neighbors, we need a mathematical function to measure the similarity (or distance) between two images. Because images are tensors, we can compare them pixel-by-pixel.

Let $I_1$ and $I_2$ be two image tensors (flattened into vectors for simplicity).

### L1 (Manhattan) Distance
The L1 distance calculates the absolute difference between individual pixels and sums them up.

$$ d_1(I_1, I_2) = \sum_{p} |I_1^p - I_2^p| $$

**Geometric Interpretation:** Imagine walking on a grid-like street network (like Manhattan). The distance is the sum of the horizontal and vertical steps. All points that have an equal L1 distance from the origin form a diamond/square shape.

### L2 (Euclidean) Distance
The L2 distance calculates the squared difference between pixels, sums them up, and takes the square root.

$$ d_2(I_1, I_2) = \sqrt{\sum_{p} (I_1^p - I_2^p)^2} $$

**Geometric Interpretation:** This is the straight-line distance ("as the crow flies"). All points that have an equal L2 distance from the origin form a perfect circle.

> [!TIP]
> **When to use which?**
> *   **L1 Distance** is highly dependent on the choice of the coordinate system. If you rotate the feature space, the L1 distance between points changes. It is better when your input features have distinct, individual meanings (e.g., height, weight, salary).
> *   **L2 Distance** is rotationally invariant. It doesn't care about the coordinate axes. It is generally preferred when features are arbitrary or interchangeable (like pixels in an image).

## 2.3 Decision Boundaries and K

When we map out the regions in space where the algorithm would predict class A vs class B, we form **Decision Boundaries**.

### The Problem with 1-Nearest Neighbor ($K=1$)
If we strictly take the single nearest neighbor, our decision boundaries will warp sharply around outlying training data points (noise). 

For example, if a single anomalous yellow dot is surrounded entirely by green dots, a $K=1$ classifier will carve out a tiny, jagged yellow island in the middle of a green ocean. This is **overfitting** to noise.

### Introducing $K > 1$
To make the classifier more robust, we increase $K$. We ask the $K$ closest points to vote. 
*   This smooths out the decision boundaries.
*   The isolated noisy yellow dot will be outvoted by its green neighbors.
*   **Drawback:** It creates "white regions" (ties) where no clear majority exists.

```mermaid
graph TD
    A[K=1] -->|Highly sensitive to noise| B(Jagged, complex boundaries)
    C[K=5] -->|Majority vote| D(Smoother, generalized boundaries)
    E[Ties] --> F[Break randomly or use distance weighting]
```

## 2.4 Why K-NN is Rarely Used for Images

Despite its pedagogical value, K-NN is almost never used for image classification in practice. 

1.  **Terrible Prediction Speed:** As discussed, $O(N)$ inference time is unacceptable.
2.  **Curse of Dimensionality:** In high-dimensional spaces (like a 3072-dimensional image vector), the concept of "distance" becomes unintuitive. The space is vast, and to densely cover it with training examples to make nearest neighbor work requires an exponentially large dataset.
3.  **Pixel Distance is Meaningless:** L1 and L2 distances on raw pixels do not correspond to perceptual similarity.
    *   Shifting an image 1 pixel to the right creates a massive L2 distance, even though the image is perceptually identical to humans.
    *   Changes in background color or slight lighting shifts dominate the pixel-wise distance, ignoring the actual semantic object.

In the next sections, we will introduce methods to automatically find the best values for $K$ and the distance metric (Hyperparameter Tuning), before moving onto more powerful parametric models.

---
### Summary of Section 2
*   **Concepts Introduced:** K-NN Algorithm, L1/L2 Distance Metrics, Decision Boundaries, Overfitting to noise, Curse of Dimensionality.
*   **Equations Derived:** L1 Distance ($d_1(I_1, I_2) = \sum |I_1 - I_2|$), L2 Distance ($d_2(I_1, I_2) = \sqrt{\sum (I_1 - I_2)^2}$).
*   **Dependencies for Next Section:** Understanding that $K$ and the distance metric are choices we must make, which naturally leads to Hyperparameter Tuning.